# Packaging the submission

Builds `<rollno>_cv_bootcamp.zip` with the layout the problem statement asks for:

```
code/          notebooks 01-06, cvlib.py, model.py, README.md
predictions/   the 10 prediction files written by notebooks 01-05
results.json   the template filled from work/results_partial.json
report.md      answers to Q1-Q5 (at most 100 words each)
```

The notebooks are taken from `MyDrive/Colab Notebooks` (where Colab keeps uploaded notebooks), so notebooks 01-05 should be saved with their outputs first. The zip is checked with `check_format.py`, saved next to the project folder and downloaded.

In [ ]:
# project folder: MyDrive/cv_bootcamp (data/, model.py, check_format.py, results_template.json, code/cvlib.py).
# If it is not there yet, it is unpacked from MyDrive/cv_bootcamp.zip, or from a zip picked in the file dialog.
import sys, zipfile
from pathlib import Path

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/cv_bootcamp")
except ImportError:                      # running outside Colab
    files = None
    ROOT = Path.cwd() / "cv_bootcamp"

if not (ROOT / "data" / "labels.csv").exists():
    zips = [z for z in [ROOT.parent / "cv_bootcamp.zip"] if z.exists()] + list(Path.cwd().glob("cv_bootcamp*.zip"))
    if not zips:
        zips = [Path(name) for name in files.upload()]
    zipfile.ZipFile(zips[0]).extractall(ROOT.parent)
    print("unpacked", zips[0].name, "into", ROOT.parent)

assert (ROOT / "data" / "labels.csv").exists() and (ROOT / "code" / "cvlib.py").exists()
sys.path.insert(0, str(ROOT / "code"))
print("project folder:", ROOT)

Mounted at /content/drive


In [ ]:
import json, re, shutil, subprocess, zipfile

ROLL_NO = "24b2428"
NAME = "Rakshana Sundaram"

WORK, PRED = ROOT / "work", ROOT / "predictions"
res = json.loads((WORK / "results_partial.json").read_text())
print(json.dumps({k: v for k, v in res.items() if k != "extra"}, indent=2))

## results.json

In [ ]:
template = json.loads((ROOT / "results_template.json").read_text())

def fill(tpl, src):
    out = {}
    for k, v in tpl.items():
        if isinstance(v, dict):
            out[k] = fill(v, src.get(k, {}))
        else:
            out[k] = src.get(k, v)
    return out

results = fill(template, res)
results["name"], results["roll_no"] = NAME, ROLL_NO.upper()
results["task4"]["methods"] = res["task4"]["methods"]          # list, copied as is
(ROOT / "results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))

## report.md

In [ ]:
x = res["extra"]
t4 = {r["method"]: r for r in x["t4_table"]}
t5 = {r["index"]: r for r in x["t5_comparison"]}
chk = x["t3_check"]

REPORT = f"""# CV Selection Bootcamp: CIFAR-10N report

Q1. The noise sits in a few look-alike pairs. Automobile/truck alone is 13.1% of all wrong labels (truck labelled automobile 1,247 times, automobile labelled truck 1,126), then cat/dog 9.7%, deer/horse 5.1% and airplane/ship 4.1%. These four pairs make up 32% of the 18,109 wrong labels. By true class, cat (51%) and deer (51%) are mislabelled most and horse (28%) least. The synthetic labels have about the same overall rate, but no single pair takes more than 2.4% of their errors.

Q2. Around epoch {res['task2']['memorization_start_epoch']}. Up to about epoch 30 the network predicts only 12 to 14% of the mislabelled training images as their wrong label, while about 65% are still predicted as their true class. After that the fit to wrong labels climbs steadily to 68% at epoch 60, and the share predicted as the true class falls to 22%. Clean validation accuracy peaks near 78% and ends at {x['noisy_ce_val_acc_clean_final']:.1%}. The climb starts as the cosine learning rate gets small. Noisy validation accuracy peaks at epoch {res['task2']['early_stop_epoch']}, used for early stopping.

Q3. Cross validated confidence over 5 folds, averaged from epoch 10 to each fold's best noisy validation epoch: AUROC {chk['cv_confidence (submitted)']['AUROC']:.3f}, precision@k {chk['cv_confidence (submitted)']['precision@k']:.3f}. Early loss from the Task 1 run was close ({chk['early_loss']['AUROC']:.3f}), but using only the last fold epoch fell to {chk['cv_last_epoch']['AUROC']:.3f}, because by then the fold models copy the annotators' mistakes. Failure mode: systematic confusions. Only 36% of trucks labelled automobile reach the top k, against over 90% for rarer errors like deer labelled dog, since the model learns the same truck/car confusion as the humans.

Q4. Co-teaching was best: noisy validation accuracy {t4['coteaching']['val acc noisy (best)']:.3f} and clean validation accuracy {t4['coteaching']['val acc clean (best)']:.1%}, against {t4['CE (Task 1, final)']['val acc clean (best)']:.1%} for plain cross entropy and {t4['CE early stop (Task 2)']['val acc clean (best)']:.1%} with early stopping. Each network updates only on the small loss part of every batch chosen by the other, so wrong labels, which keep a high loss early on, are mostly skipped, and the two networks do not reinforce each other's errors. Its accuracy kept rising until epoch 60 with no memorisation dip. Relabelling with the Task 3 scores came close ({t4['relabel']['val acc clean (best)']:.1%}) after cutting label noise from 40% to {x['relabel_noise_after']:.0%}.

Q5. No. On both label sets the order is co-teaching, relabel, SCE, early stopped CE, final CE (clean validation accuracy {t5['coteaching']['synthetic noise']:.1%}, {t5['relabel']['synthetic noise']:.1%}, {t5['sce']['synthetic noise']:.1%}, {t5['CE early stop']['synthetic noise']:.1%}, {t5['CE final']['synthetic noise']:.1%} on synthetic against {t5['coteaching']['human noise']:.1%}, {t5['relabel']['human noise']:.1%}, {t5['sce']['human noise']:.1%}, {t5['CE early stop']['human noise']:.1%}, {t5['CE final']['human noise']:.1%} on human noise). Every method does better on synthetic noise. Random wrong labels share no pattern, so they are fitted later and less: at epoch {res['task2']['early_stop_epoch']} the network fits 4.8% of them against 17.6% of human ones, and early loss detects them with AUROC {x['t5_early_loss_auroc']['synthetic']:.3f} against {x['t5_early_loss_auroc']['human']:.3f}. Human mistakes are plausible, so they are harder to separate.
"""

# word count the way check_format.py does it (question line included)
for q, body in re.findall(r"^(Q\d)\.(.*?)(?=^Q\d\.|\Z)", REPORT, flags=re.M | re.S):
    print(q, len(body.split()), "words")
_ = (ROOT / "report.md").write_text(REPORT)

## code/README.md

In [ ]:
README = """# CV Selection Bootcamp: learning from noisy labels (CIFAR-10N)

Everything runs on Google Colab with an A100 GPU. Seed 42 for every run; cuDNN autotuning is on, so reruns can differ in the last decimals.

## Setup

1. Put a folder `cv_bootcamp` in Google Drive (`MyDrive/cv_bootcamp`) containing `data/` (the provided files), `model.py`, `check_format.py`, `results_template.json` and `code/cvlib.py`.
2. Open each notebook in Colab, choose an A100 runtime and use Runtime > Run all, in the order below. Each notebook mounts Drive and reads and writes inside `MyDrive/cv_bootcamp` (`work/` for intermediate files, `predictions/` for the submission files). Finished training runs are saved in `work/` and are loaded instead of retrained on a rerun.

## Notebooks

| Notebook | Task | Writes | Time (A100) |
|---|---|---|---|
| `01_task1_noise_and_clean_ref_final.ipynb` | Task 1: noise rate, confusion matrix, clean-label reference | `t1_clean_ref.csv` | 5 min |
| `02_task1_noisy_ce_task2_dynamics_final.ipynb` | Task 1 noisy CE run, Task 2 dynamics and early stopping | `t1_noisy_ce.csv`, `t2_dynamics.csv`, `t2_early_stop.csv` | 6 min |
| `03_task3_noise_scores_final.ipynb` | Task 3: 5-fold cross-validated noise scores and the clean-label check | `t3_noise_scores.csv` | 25 min |
| `04_task4_robust_training_final.ipynb` | Task 4: relabel, SCE, co-teaching | `t4_relabel.csv`, `t4_sce.csv`, `t4_coteaching.csv` | 20 min |
| `05_task5_synthetic_noise_final.ipynb` | Task 5: CE and co-teaching on synthetic labels, comparison | `t5_baseline_synth.csv`, `t5_best_synth.csv` | 45 min |
| `06_package_submission_final.ipynb` | results.json, report.md, this README, the zip | `<rollno>_cv_bootcamp.zip` | 1 min |

`cvlib.py` holds the shared code: data loading, GPU augmentation (random crop with 4-pixel padding, horizontal flip), the training loop (SGD with Nesterov momentum 0.9, lr 0.1 with per-step cosine decay, weight decay 5e-4, batch 128, 60 epochs, bf16 autocast), symmetric cross-entropy, co-teaching and the prediction writers. `model.py` is the provided ResNet-18, trained from scratch everywhere.

## Rules followed

* Clean labels are used only to train the Task 1 reference and to measure (confusion matrix, Task 2 curves, Task 3 check, clean validation accuracy printed next to results). Every early-stopping epoch, averaging window, method choice and checkpoint comes from noisy validation accuracy (synthetic validation accuracy in Task 5).
* Budget: 60 epochs per network; co-teaching uses two networks; Task 3 uses 5 runs. In Task 5 the co-teaching noise-rate estimate repeats the Task 3 cross-validation on the synthetic labels, and relabel and SCE are also run on synthetic labels only for the ranking in Q5 (not submitted).

Notebook code was written with help from an AI assistant (Claude) and checked and run by me.
"""
(ROOT / "code").mkdir(exist_ok=True)
(ROOT / "code" / "README.md").write_text(README)
print("README written")

## Assemble and check the zip

In [ ]:
NAMES = {
    "01": "01_task1_noise_and_clean_ref_final.ipynb",
    "02": "02_task1_noisy_ce_task2_dynamics_final.ipynb",
    "03": "03_task3_noise_scores_final.ipynb",
    "04": "04_task4_robust_training_final.ipynb",
    "05": "05_task5_synthetic_noise_final.ipynb",
    "06": "06_package_submission_final.ipynb",
}
search = [ROOT.parent / "Colab Notebooks", ROOT, ROOT / "code"]
found = {}
for d in search:
    for p in d.glob("0?_*.ipynb") if d.exists() else []:
        k = p.name[:2]
        if k in NAMES and (k not in found or p.stat().st_mtime > found[k].stat().st_mtime):
            found[k] = p
for k in NAMES:
    print(k, found.get(k, "NOT FOUND"))

SUB = ROOT / "submission" / f"{ROLL_NO}_cv_bootcamp"
shutil.rmtree(SUB.parent, ignore_errors=True)
(SUB / "code").mkdir(parents=True)
(SUB / "predictions").mkdir()
for k, p in found.items():
    shutil.copy(p, SUB / "code" / NAMES[k])
for f in ["cvlib.py", "README.md"]:
    shutil.copy(ROOT / "code" / f, SUB / "code" / f)
shutil.copy(ROOT / "model.py", SUB / "code" / "model.py")

wanted = ["t1_clean_ref.csv", "t1_noisy_ce.csv", "t2_dynamics.csv", "t2_early_stop.csv", "t3_noise_scores.csv",
          "t5_baseline_synth.csv", "t5_best_synth.csv"] + [m["file"] for m in results["task4"]["methods"]]
for f in wanted:
    shutil.copy(PRED / f, SUB / "predictions" / f)
shutil.copy(ROOT / "results.json", SUB / "results.json")
shutil.copy(ROOT / "report.md", SUB / "report.md")

ZIP = ROOT.parent / f"{ROLL_NO}_cv_bootcamp.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(SUB.parent))
print(ZIP, f"{ZIP.stat().st_size / 1e6:.1f} MB")

out = subprocess.run([sys.executable, "check_format.py", str(ZIP)], cwd=ROOT, capture_output=True, text=True)
print(out.stdout, out.stderr)

In [ ]:
# download the zip to the computer as well (it is also saved in MyDrive)
try:
    files.download(str(ZIP))
except Exception as e:
    print("download skipped:", e)